
# Text Analysis in Julia, with [TextAnalysis.jl]( https://juliatext.github.io/TextAnalysis.jl/dev/ )  
### Aliza Rogers


#### TextAnalysis.jl is a package that is designed to make NLP easier. It relieves some burden of preprocessing and creating documents and a corpus. 

In [2]:
using TextAnalysis

<br>
To begin, we will need documents of text to work with. 

Below is a function, that may be used to load in documents. It will read the plain text file at the path name that was given and return a StringDocument. 

This function works particularly well with Project Gutenberg books, as it strips all information before and after the start/end markers.
<br>

In [3]:
# Please note that this function assumes that all documents are plain text and UTF-8 encoded

function get_document(path::String)
    #reading in the file from the given path
    text = read(path, String)
    
    # Project Gutenberg files typically start and end with these markers
    start_regex = r"\*\*\* START OF (?:THIS|THE) PROJECT GUTENBERG EBOOK.*?\*\*\*"
    end_regex = r"\*\*\* END OF (?:THIS|THE) PROJECT GUTENBERG EBOOK.*?\*\*\*"

    start_match = findfirst(start_regex, text)
    end_match = findfirst(end_regex, text)

    if !isnothing(start_match) && !isnothing(end_match)
        
        # Convert byte indices to valid character indices
        start_idx = nextind(text, start_match.stop - 1)
        end_idx = prevind(text, end_match.start + 1)

        text = String(SubString(text, start_idx, end_idx))
    end

    return StringDocument(text)
end

get_document (generic function with 1 method)

<br>

All ebooks that I will use are from [Project Gutenberg]( https://www.gutenberg.org/ ). This is a libary of free e-books that are in the United States public domain. 

I will prepare works of fiction that were written for children/young adults. Keeping the type of document the same is important for when we will create a corpus.
<br>

In [4]:
# getting the path to the directory containing the Project Gutenberg books
book_path = pwd() * "/ebooks"


#Alice's Adventures in Wonderland - Lewis Carroll
wonderland = get_document(book_path * "/11-0.txt")

title!(wonderland, "Alice's Adventures in Wonderland")
author!(wonderland, "Lewis Carroll")
timestamp!(wonderland, "1865")


# The Wonderful Wizard of Oz - L. Frank Baum
oz = get_document(book_path * "/55-0.txt")

title!(oz, "The Wonderful Wizard of Oz")
author!(oz, "L. Frank Baum")
timestamp!(oz, "1900")


# The Secret Garden - Frances Hodgson Burnett
secret_garden = get_document(book_path * "/113-0.txt")

title!(secret_garden, "The Secret Garden")
author!(secret_garden, "Frances Hodgson Burnett")
timestamp!(secret_garden, "1911")


# Treasure Island - Robert Louis Stevenson
treasure_island = get_document(book_path * "/120-0.txt")

title!(treasure_island, "Treasure Island")
author!(treasure_island, "Robert Louis Stevenson")
timestamp!(treasure_island, "1883")


# The Jungle Book - Rudyard Kipling
jungle_book = get_document(book_path * "/236-0.txt")

title!(jungle_book, "The Jungle Book")
author!(jungle_book, "Rudyard Kipling")
timestamp!(jungle_book, "1894")


# The Adventures of Pinocchio - C. Collodi
pinocchio = get_document(book_path * "/500-0.txt")

title!(pinocchio, "The Adventures of Pinocchio")
author!(pinocchio, "C. Collodi")
timestamp!(pinocchio, "1883")


doc_list = [wonderland, oz, secret_garden, treasure_island, jungle_book, pinocchio]

6-element Vector{StringDocument{String}}:
 A StringDocument{String}
 A StringDocument{String}
 A StringDocument{String}
 A StringDocument{String}
 A StringDocument{String}
 A StringDocument{String}

Let's take a quick look at the text of Pinocchio, just to be sure that the code did what was expected.

In [5]:
text(pinocchio)

"*\r\n\r\n\r\n\r\n\r\nProduced by Charles Keller (for Tina); and David Widger\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\nDashes; small checks; quick pass; gutchecked twice; jeebies; spellcheck\r\n\r\n\r\nTHE ADVENTURES OF PINOCCHIO\r\n\r\nby C. Collodi\r\n\r\n[Pseudonym of Carlo Lorenzini]\r\n\r\n\r\nTranslated from the Italia" ⋯ 225379 bytes ⋯ " said to himself with great content:\r\n\r\n“How ridiculous I was as a Marionette! And how happy I am, now that I\r\nhave become a real boy!”\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\nEnd of the Project Gutenberg EBook of The Adventures of Pinocchio, by\r\nC. Collodi--Pseudonym of Carlo Lorenzini\r\n\r\n*"

<p> Next, we will create a corpus. A corpus is a collection of documents, to be analyzed. The most common words within the corpus will be displayed between each step of cleaning it up. This will be very useful for understanding <i>why</i> clean up is important.</p>

In [6]:
corpus = Corpus(doc_list)

using OrderedCollections

update_lexicon!(corpus)

lexicon(corpus)

print("These are the most common words in our corpus, without any clean up:")

ordered_lexicon = OrderedDict(sort(collect(lexicon(corpus)), by = x -> x[2], rev = true))

These are the most common words in our corpus, without any clean up:

OrderedDict{String, Int64} with 18247 entries:
  ","    => 21651
  "the"  => 16002
  "and"  => 11585
  "“"    => 8078
  "to"   => 7676
  "’"    => 7180
  "a"    => 6849
  "”"    => 6708
  "of"   => 6055
  "I"    => 5957
  "he"   => 4307
  "was"  => 4220
  "in"   => 3774
  "that" => 3172
  "you"  => 2990
  "it"   => 2911
  "his"  => 2850
  "as"   => 2608
  "had"  => 2583
  "!"    => 2520
  "she"  => 2377
  "said" => 2335
  "with" => 2226
  "s"    => 2128
  "for"  => 2018
  ⋮      => ⋮

<b> Notice that quite a few of the most popular words are punctuation marks. To fix this, we can clean up the corpus by removing punctuation.</b>

In [7]:
prepare!(corpus, strip_punctuation)

remove_case!(corpus) # making all letters lowercase

print("These are the most common words in our corpus, with the punctuation and casing removed:")
update_lexicon!(corpus)
ordered_lexicon = OrderedDict(sort(collect(lexicon(corpus)), by = x -> x[2], rev = true))


These are the most common words in our corpus, with the punctuation and casing removed:

OrderedDict{String, Int64} with 13675 entries:
  "the"  => 17343
  "and"  => 12220
  "to"   => 7759
  "a"    => 7014
  "of"   => 6143
  "he"   => 5173
  "i"    => 5167
  "was"  => 4267
  "in"   => 4017
  "it"   => 3768
  "that" => 3336
  "you"  => 3334
  "his"  => 2932
  "she"  => 2904
  "as"   => 2774
  "said" => 2600
  "had"  => 2596
  "with" => 2299
  "for"  => 2128
  "on"   => 2002
  "at"   => 1983
  "but"  => 1956
  "her"  => 1822
  "they" => 1761
  "him"  => 1758
  ⋮      => ⋮

<b> These words are very general, and they don't tell much about the text. In NLP, these are called "stop words". Stop words usually include articles, prepositions, pronouns, and other similar words. </b>

In [8]:
prepare!(corpus, strip_stopwords)

print("These are the most common words in our corpus, without punctuation, casing, and stop words:")
update_lexicon!(corpus)
ordered_lexicon = OrderedDict(sort(collect(lexicon(corpus)), by = x -> x[2], rev = true))

These are the most common words in our corpus, without punctuation, casing, and stop words:

OrderedDict{String, Int64} with 13272 entries:
  "little"    => 994
  "mary"      => 673
  "time"      => 531
  "head"      => 491
  "looked"    => 482
  "pinocchio" => 430
  "eyes"      => 388
  "alice"     => 386
  "dont"      => 382
  "th"        => 372
  "look"      => 370
  "dorothy"   => 347
  "ill"       => 335
  "answered"  => 328
  "cried"     => 313
  "day"       => 312
  "found"     => 311
  "heard"     => 307
  "colin"     => 302
  "tell"      => 296
  "door"      => 289
  "im"        => 283
  "dickon"    => 277
  "round"     => 277
  "garden"    => 256
  ⋮           => ⋮

Now, these common words make more sense. Quite a few are character names, which are expected to be repeated.

It is important to note that these raw word counts do not account for works being different lengths. Take a look at the documents' word counts.

In [9]:
using Printf

tokenized_doc_list = []

for doc in doc_list
   token_doc = TokenDocument(text(doc))
   push!(tokenized_doc_list, token_doc)
   @printf("Document: %-35s Word Count: %d\n", title(doc), length(tokens(token_doc)))
end


Document: Alice's Adventures in Wonderland    Word Count: 8876
Document: The Wonderful Wizard of Oz          Word Count: 13306
Document: The Secret Garden                   Word Count: 27786
Document: Treasure Island                     Word Count: 24184
Document: The Jungle Book                     Word Count: 19319
Document: The Adventures of Pinocchio         Word Count: 14202


Let's take a look at how removing <i>The Secret Garden</i> would effect the most frequent words in our corpus. We will have to clean up the corpus again, as those functions do not edit the documents themselves.

In [10]:
non_garden_corpus = Corpus([wonderland, oz, treasure_island, jungle_book, pinocchio])

A Corpus with 5 documents:
 * 5 StringDocument's
 * 0 FileDocument's
 * 0 TokenDocument's
 * 0 NGramDocument's

Corpus's lexicon contains 0 tokens
Corpus's index contains 0 tokens

In [11]:
prepare!(non_garden_corpus, strip_punctuation)
remove_case!(non_garden_corpus)
prepare!(non_garden_corpus, strip_stopwords)
update_lexicon!(non_garden_corpus)
ordered_lexicon = OrderedDict(sort(collect(lexicon(non_garden_corpus)), by = x -> x[2], rev = true))

OrderedDict{String, Int64} with 11253 entries:
  "little"     => 772
  "pinocchio"  => 430
  "time"       => 413
  "head"       => 406
  "alice"      => 386
  "dorothy"    => 347
  "dont"       => 270
  "looked"     => 258
  "eyes"       => 255
  "cried"      => 251
  "ill"        => 241
  "silver"     => 237
  "scarecrow"  => 220
  "found"      => 219
  "sea"        => 219
  "heard"      => 212
  "captain"    => 212
  "day"        => 211
  "voice"      => 208
  "poor"       => 203
  "soon"       => 202
  "mowgli"     => 202
  "marionette" => 199
  "look"       => 199
  "tell"       => 193
  ⋮            => ⋮

Notice that "Mary" and "Colin" are no longer frequent words. These are two main characters in <i>The Secret Garden</i>.